# 16 · Diffusion: DDPM training and DDPM / DDIM sampling

**Needed for:** DDPM (vision track, step V3), then DiT and flow matching (notebook 17), and image generation in
Janus-style models.

**Forward process (fixed):** add Gaussian noise over $T$ steps until the data is pure noise. Thanks to Gaussians
adding up, you can jump to any step directly:
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \bar\alpha_t = \prod_{s\le t}(1-\beta_s)$$

**Training:** pick a random $t$, noise the data, and train a network $\epsilon_\theta(x_t, t)$ to predict the noise
$\epsilon$. Plain MSE.

**Sampling:** start from pure noise and step backwards using the predicted noise. DDPM takes all $T$ steps with
fresh noise each time; DDIM is deterministic and can skip most steps.

Everything runs on 2-D toy data (a ring of 8 Gaussians) so it trains in seconds.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Data and noise schedule

In [ ]:
CENTERS = torch.stack([torch.tensor([math.cos(2 * math.pi * i / 8), math.sin(2 * math.pi * i / 8)]) * 2 for i in range(8)])


def sample_data(n):
    labels = torch.randint(0, 8, (n,))
    return CENTERS[labels] + 0.1 * torch.randn(n, 2), labels


def mode_quality(x):
    """Fraction of samples within 0.3 of some center, and how many of the 8 modes got at least 2% of samples."""
    d = torch.cdist(x.cpu(), CENTERS)
    near = d.min(1).values < 0.3
    counts = torch.bincount(d.argmin(1)[near], minlength=8)
    return near.float().mean().item(), (counts > 0.02 * len(x)).sum().item()


T = 200
betas = torch.linspace(1e-4, 0.05, T)       # short schedule for a toy problem; images typically use T = 1000
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, 0)
print(f"alpha_bar: t=0 {alpha_bar[0]:.4f}   t=T/2 {alpha_bar[T // 2]:.4f}   t=T-1 {alpha_bar[-1]:.6f} (almost pure noise)")

## 2. `q_sample` and a check of its statistics

In [ ]:
def q_sample(x0, t, noise, alpha_bar):
    """x0: (B, ...), t: (B,) long, noise like x0."""
    ab = alpha_bar[t].view(-1, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise


def test_q_sample(fn):
    torch.manual_seed(0)
    x0 = torch.full((200_000, 2), 3.0)
    t = torch.full((200_000,), 50)
    xt = fn(x0, t, torch.randn_like(x0), alpha_bar)
    torch.testing.assert_close(xt.mean(0), torch.full((2,), 3.0 * alpha_bar[50].sqrt().item()), atol=0.01, rtol=0)
    torch.testing.assert_close(xt.var(0), torch.full((2,), 1 - alpha_bar[50].item()), atol=0.01, rtol=0)


test_q_sample(q_sample)
print("q_sample mean and variance match the closed form")

## 3. The noise-prediction network and training

The network needs to know $t$. Standard trick: sinusoidal embedding of $t$ (like transformer positions), passed
through an MLP and added to the hidden state.

In [ ]:
def timestep_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.cos(), args.sin()], -1)


class EpsNet(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(64, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.inp = nn.Linear(2, dim)
        self.body = nn.Sequential(nn.SiLU(), nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, 2))

    def forward(self, x, t):
        return self.body(self.inp(x) + self.t_mlp(timestep_embedding(t, 64)))


torch.manual_seed(0)
model = EpsNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
ab_dev = alpha_bar.to(device)
for step in range(3001):
    x0, _ = sample_data(512)
    x0 = x0.to(device)
    t = torch.randint(0, T, (512,), device=device)
    noise = torch.randn_like(x0)
    loss = F.mse_loss(model(q_sample(x0, t, noise, ab_dev), t), noise)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}")

## 4. DDPM sampling (ancestral)

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big) + \sigma_t z, \qquad \sigma_t^2 = \beta_t,\ z\sim\mathcal N(0, I)$$

No noise is added at the last step ($t = 0$).

In [ ]:
def ddpm_step(x_t, t, eps, betas, alphas, alpha_bar, noise):
    """One reverse step for integer t (same t for the batch)."""
    mean = (x_t - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
    return mean if t == 0 else mean + betas[t].sqrt() * noise


@torch.no_grad()
def sample_ddpm(model, n):
    x = torch.randn(n, 2, device=device)
    b, a, ab = betas.to(device), alphas.to(device), alpha_bar.to(device)
    for t in reversed(range(T)):
        eps = model(x, torch.full((n,), t, device=device))
        x = ddpm_step(x, t, eps, b, a, ab, torch.randn_like(x))
    return x


def test_ddpm_step(fn):
    torch.manual_seed(0)
    x0 = torch.randn(4, 2)
    eps = torch.randn(4, 2)
    t = 120
    x_t = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * eps
    # with the true noise, the posterior mean is a known mix of x0 and x_t
    coef0 = alpha_bar[t - 1].sqrt() * betas[t] / (1 - alpha_bar[t])
    coef_t = alphas[t].sqrt() * (1 - alpha_bar[t - 1]) / (1 - alpha_bar[t])
    expected = coef0 * x0 + coef_t * x_t
    torch.testing.assert_close(fn(x_t, t, eps, betas, alphas, alpha_bar, torch.zeros(4, 2)), expected)
    torch.testing.assert_close(fn(x_t, 0, eps, betas, alphas, alpha_bar, torch.randn(4, 2)),
                               fn(x_t, 0, eps, betas, alphas, alpha_bar, torch.zeros(4, 2)))  # no noise at t = 0


test_ddpm_step(ddpm_step)
frac, modes = mode_quality(sample_ddpm(model, 4000))
print(f"DDPM, {T} steps: {frac:.0%} of samples near a mode, {modes}/8 modes covered")

## 5. DDIM: deterministic, fewer steps

Predict $\hat x_0$ from the current noise estimate, then jump directly to an earlier step $s < t$:
$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta}{\sqrt{\bar\alpha_t}}, \qquad x_s = \sqrt{\bar\alpha_s}\,\hat x_0 + \sqrt{1-\bar\alpha_s}\,\epsilon_\theta$$
Same trained model, no retraining. Fewer steps trade some quality for speed.

In [ ]:
@torch.no_grad()
def sample_ddim(model, n, steps):
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    ab = alpha_bar.to(device)
    x = torch.randn(n, 2, device=device)
    for i, t in enumerate(ts):
        eps = model(x, torch.full((n,), t, device=device))
        x0_hat = (x - (1 - ab[t]).sqrt() * eps) / ab[t].sqrt()
        if i == len(ts) - 1:
            return x0_hat
        s = ts[i + 1]
        x = ab[s].sqrt() * x0_hat + (1 - ab[s]).sqrt() * eps


for steps in (10, 25, 50):
    frac, modes = mode_quality(sample_ddim(model, 4000, steps))
    print(f"DDIM, {steps:2d} steps: {frac:.0%} near a mode, {modes}/8 modes covered")

## Things to be able to explain

- Predicting $\epsilon$, $x_0$, or velocity $v = \sqrt{\bar\alpha}\epsilon - \sqrt{1-\bar\alpha}x_0$ are
  equivalent reparameterizations. Why might one train better at high or low noise levels?
- Why does a linear β schedule waste steps on images, and what does the cosine schedule change?
- Latent diffusion (Stable Diffusion): run all of this on a VAE's latent grid instead of pixels. Why is that the
  big practical win?

## Drill

In [ ]:
def my_q_sample(x0, t, noise, alpha_bar):
    raise NotImplementedError


def my_ddpm_step(x_t, t, eps, betas, alphas, alpha_bar, noise):
    raise NotImplementedError


drill(test_q_sample, my_q_sample)
drill(test_ddpm_step, my_ddpm_step)